In [1]:

import glob
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
from fugashi import Tagger


# ==========================================
# 1. 形態素解析の設定
# ==========================================

tagger = Tagger()

FUNCTION_POS = {"助詞", "助動詞", "接続詞", "連体詞"}
CONTENT_POS = {"名詞", "動詞", "形容詞", "形状詞", "副詞", "感動詞"}


# ==========================================
# 2. Hapax率・Yule's K
# ==========================================

def hapax_yule(tokens, word_count):

    if word_count == 0:
        return 0, 0

    freq = Counter(tokens)

    # Hapax率
    hapax_count = sum(1 for v in freq.values() if v == 1)
    hapax_ratio = hapax_count / word_count

    # Yule's K
    freq_counter = Counter(freq.values())
    m1 = word_count
    m2 = sum((f ** 2) * c for f, c in freq_counter.items())

    yules_k = (
        10000 * ((m2 - m1) / (m1 * m1))
        if m1 > 0 else 0
    )

    return hapax_ratio, yules_k


# ==========================================
# 3. スタイロメトリ特徴量抽出
# ==========================================

def analyze_text_files(folder_path):

    search_path = os.path.join(folder_path, "*.txt")

    filepaths = sorted(
        glob.glob(search_path),
        key=lambda x: int(re.search(r"\d+", os.path.basename(x)).group())
        if re.search(r"\d+", os.path.basename(x))
        else 0
    )

    feature_data = []

    for filepath in filepaths:

        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        pos_counter = Counter()
        particle_counter = Counter()

        all_words = []
        function_words = []
        content_words = []

        word_count = 0
        function_count = 0
        content_count = 0

        # ======================================
        # 文字・記号特徴
        # ======================================

        comma_count = text.count("、")
        period_count = text.count("。")
        exclamation_count = text.count("！")
        question_count = text.count("？")

        sentences = re.split(r"[。！？]", text)

        sentence_lengths = [
            len(s.strip())
            for s in sentences
            if s.strip()
        ]

        avg_sentence_length = (
            np.mean(sentence_lengths)
            if sentence_lengths else 0
        )

        sentence_std = (
            np.std(sentence_lengths)
            if sentence_lengths else 0
        )

        max_sentence_length = (
            max(sentence_lengths)
            if sentence_lengths else 0
        )

        total_chars = len(text)

        hiragana_count = len(re.findall(r"[ぁ-ん]", text))
        katakana_count = len(re.findall(r"[ァ-ン]", text))
        kanji_count = len(re.findall(r"[一-龥]", text))
        alnum_count = len(re.findall(r"[a-zA-Z0-9]", text))

        hiragana_ratio = hiragana_count / total_chars if total_chars else 0
        katakana_ratio = katakana_count / total_chars if total_chars else 0
        kanji_ratio = kanji_count / total_chars if total_chars else 0
        alnum_ratio = alnum_count / total_chars if total_chars else 0

        # ======================================
        # 形態素解析
        # ======================================

        for word in tagger(text):

            pos = word.feature.pos1
            surface = word.surface

            if pos == "補助記号":
                continue

            if pos == "助詞":
                particle_counter[surface] += 1

            all_words.append(surface)
            pos_counter[pos] += 1
            word_count += 1

            if pos in FUNCTION_POS:
                function_words.append(surface)
                function_count += 1

            elif pos in CONTENT_POS:
                content_words.append(surface)
                content_count += 1

        # ======================================
        # 基本特徴量
        # ======================================

        if word_count > 0:

            function_ratio = function_count / word_count * 100
            content_ratio = content_count / word_count * 100

            comma_ratio = comma_count / word_count
            period_ratio = period_count / word_count

            ttr = len(set(all_words)) / word_count

            # Hapax / Yule
            hapax_all, yule_all = hapax_yule(
                all_words, word_count
            )

            hapax_func, yule_func = hapax_yule(
                function_words, function_count
            )

            hapax_cont, yule_cont = hapax_yule(
                content_words, content_count
            )

        else:

            function_ratio = 0
            content_ratio = 0
            comma_ratio = 0
            period_ratio = 0
            ttr = 0

            hapax_all = yule_all = 0
            hapax_func = yule_func = 0
            hapax_cont = yule_cont = 0

        # ======================================
        # 助詞割合
        # ======================================

        total_particle = sum(particle_counter.values())

        particle_ratio = {
            f"助詞_{k}": (
                v / total_particle
                if total_particle > 0 else 0
            )
            for k, v in particle_counter.items()
        }

        # ======================================
        # 品詞割合
        # ======================================

        pos_ratio = {
            pos: count / word_count * 100
            for pos, count in pos_counter.items()
        } if word_count > 0 else {}

        # ======================================
        # 特徴量保存
        # ======================================

        feature_data.append({

            "filename": os.path.basename(filepath),

            "総単語数": word_count,
            "機能語率": function_ratio,
            "内容語率": content_ratio,

            "異なり語数": len(set(all_words)),
            "TTR": ttr,

            "ひらがな率": hiragana_ratio,
            "カタカナ率": katakana_ratio,
            "漢字率": kanji_ratio,
            "英数字率": alnum_ratio,

            "文長標準偏差": sentence_std,
            "最大文長": max_sentence_length,
            "平均文長": avg_sentence_length,

            "Hapax率_all": hapax_all,
            "Yules_K_all": yule_all,

            "Hapax率_func": hapax_func,
            "Yules_K_func": yule_func,

            "Hapax率_cont": hapax_cont,
            "Yules_K_cont": yule_cont,

            "名詞割合": pos_ratio.get("名詞", 0),
            "動詞割合": pos_ratio.get("動詞", 0),
            "形容詞割合": pos_ratio.get("形容詞", 0),
            "形状詞割合": pos_ratio.get("形状詞", 0),
            "助詞割合": pos_ratio.get("助詞", 0),
            "助動詞割合": pos_ratio.get("助動詞", 0),
            "副詞割合": pos_ratio.get("副詞", 0),
            "連体詞割合": pos_ratio.get("連体詞", 0),
            "接続詞割合": pos_ratio.get("接続詞", 0),
            "感動詞割合": pos_ratio.get("感動詞", 0),

            **particle_ratio,

            "読点頻度": comma_ratio,
            "句点頻度": period_ratio,
            "感嘆符数": exclamation_count,
            "疑問符数": question_count,

        })

    return feature_data


# ==========================================
# 4. 学習用・テスト用データの抽出
# ==========================================

if __name__ == "__main__":

    base_path = r"C:\Users\Admin\2026-"

    # 学習用データ
    train_data = analyze_text_files(
        os.path.join(base_path, "train_Qwen_text")
    )

    train_df = pd.DataFrame(train_data)

    train_df.to_csv(
        "train_Qwen_stylometry_features.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # テスト用データ
    test_data = analyze_text_files(
        os.path.join(base_path, "test_Qwen_text")
    )

    test_df = pd.DataFrame(test_data)

    test_df.to_csv(
        "test_Qwen_stylometry_features.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("=== 学習用データ ===")
    print(train_df.head())
    print(f"総データ数：{len(train_df)}")

    print("\n=== テスト用データ ===")
    print(test_df.head())
    print(f"総データ数：{len(test_df)}")

    # ======================================
    # 結果表示
    # ======================================

    print("=== 学習用データ ===")
    print(train_df.head())
    print(f"総データ数：{len(train_df)}")

    print("\n=== テスト用データ ===")
    print(test_df.head())
    print(f"総データ数：{len(test_df)}")


=== 学習用データ ===
                 filename  総単語数       機能語率       内容語率  異なり語数       TTR  \
0  train_Qwen_text_01.txt   270  42.962963  50.740741    149  0.551852   
1  train_Qwen_text_02.txt   270  40.740741  56.296296    148  0.548148   
2  train_Qwen_text_03.txt   279  42.652330  52.329749    150  0.537634   
3  train_Qwen_text_04.txt   285  44.210526  50.526316    147  0.515789   
4  train_Qwen_text_05.txt   301  43.189369  49.833887    155  0.514950   

      ひらがな率     カタカナ率       漢字率      英数字率  ...  疑問符数      助詞_へ     助詞_つつ  \
0  0.426692  0.182331  0.278195  0.011278  ...     0       NaN       NaN   
1  0.389098  0.127820  0.360902  0.018797  ...     0  0.013514       NaN   
2  0.459959  0.071869  0.396304  0.000000  ...     0       NaN       NaN   
3  0.477642  0.012195  0.441057  0.000000  ...     0  0.011111       NaN   
4  0.485549  0.034682  0.404624  0.000000  ...     0  0.010753  0.010753   

       助詞_か  助詞_ながら  助詞_だけ  助詞_ば  助詞_ほど  助詞_しも  助詞_より  
0       NaN     NaN    NaN 